# 🧹 Data Cleaning & Preprocessing — Internship Application Data

**Objective:** Ensure data accuracy by cleaning and transforming raw internship application datasets.

**What we will do:**
- Generate intentionally messy/raw data
- Identify and handle missing values
- Remove duplicates
- Detect and treat outliers
- Standardize and structure data for analysis
- Automate the full pipeline using Python (Pandas)

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## Step 2: Generate Raw (Messy) Dataset

In [ ]:
np.random.seed(42)
n = 200

departments   = ['Tech', 'HR', 'Marketing', 'Finance', 'Operations', 'tech', 'hr', 'MARKETING']
genders       = ['Male', 'Female', 'male', 'female', 'M', 'F', None]
education     = ['Bachelor', 'Master', 'PhD', 'bachelor', 'MASTER', 'phd', None]
cities        = ['Karachi', 'Lahore', 'Islamabad', 'karachi', 'lahore', 'ISLAMABAD', None]

raw_data = {
    'application_id':      [f'APP{1000+i}' for i in range(n)],
    'applicant_name':      [f'  Applicant_{i}  ' for i in range(n)],   # extra spaces
    'age':                 list(np.random.randint(21, 30, n-10)) + [150, -5, 999, 17, 200, None, None, None, None, None],  # outliers + nulls
    'gender':              np.random.choice(genders, n).tolist(),
    'city':                np.random.choice(cities, n).tolist(),
    'department':          np.random.choice(departments, n).tolist(),
    'education_level':     np.random.choice(education, n).tolist(),
    'gpa':                 list(np.round(np.random.uniform(2.0, 4.0, n-8), 2)) + [5.5, -1.0, 9.9, 0.0, None, None, None, None],  # bad values
    'experience_years':    list(np.random.randint(0, 5, n-5)) + [None, None, None, -3, 99],
    'num_skills':          list(np.random.randint(2, 12, n-5)) + [None, None, 100, -1, None],
    'expected_salary':     list(np.random.randint(30000, 80000, n-5)) + [None, None, None, 9999999, -5000],
    'phone_number':        ['03' + str(np.random.randint(100000000, 999999999)) for _ in range(n-3)] + ['abc', None, '123'],
    'application_date':    pd.date_range('2024-01-01', periods=n-5).tolist() + [None, None, None, '99-99-9999', 'not a date'],
}

df_raw = pd.DataFrame(raw_data)

# Inject 20 duplicate rows
duplicates = df_raw.sample(20, random_state=1)
df_raw = pd.concat([df_raw, duplicates], ignore_index=True)

print(f'Raw dataset created: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')
print(f'   (Includes missing values, duplicates, outliers, inconsistent formatting)')
df_raw.head(10)

## Step 3: Initial Data Inspection

In [ ]:
print('Dataset Shape:', df_raw.shape)
print()
print('Data Types:')
print(df_raw.dtypes)
print()
print('First look:')
df_raw.info()

In [ ]:
# Missing Values Summary
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('Missing Values Report:')
print(missing_df)

# Visualize
plt.figure(figsize=(10, 4))
missing_df['Missing %'].plot(kind='bar', color='#e74c3c', edgecolor='white')
plt.title('Missing Values by Column (%)', fontsize=13, fontweight='bold')
plt.ylabel('Missing %')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Duplicate Check
dup_count = df_raw.duplicated().sum()
print(f'Duplicate rows found: {dup_count}')
print()
print('Basic Statistics:')
df_raw.describe().round(2)

## Step 4: Remove Duplicates

In [ ]:
df = df_raw.copy()

before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f'Duplicates removed: {before - after} rows')
print(f'   Rows before: {before} → Rows after: {after}')

## Step 5: Standardize Text Columns

In [ ]:
# Before standardization
print('Before - Department unique values:')
print(df['department'].value_counts().to_dict())

# Strip whitespace + Title Case
text_cols = ['applicant_name', 'gender', 'city', 'department', 'education_level']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
    df[col] = df[col].replace('None', np.nan)
    df[col] = df[col].replace('Nan', np.nan)

# Fix gender abbreviations
df['gender'] = df['gender'].replace({'M': 'Male', 'F': 'Female'})

print()
print('After - Department unique values:')
print(df['department'].value_counts().to_dict())
print()
print('Text columns standardized!')

## Step 6: Handle Missing Values

In [ ]:
# Numeric columns → fill with median
num_cols = ['age', 'gpa', 'experience_years', 'num_skills', 'expected_salary']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f'   {col}: filled with median = {median_val}')

print()

# Categorical columns → fill with mode
cat_cols = ['gender', 'city', 'department', 'education_level']
for col in cat_cols:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)
    print(f'   {col}: filled with mode = {mode_val}')

print()
print(f' Missing values after treatment: {df.isnull().sum().sum()}')

## Step 7: Fix Invalid Values & Outliers

In [ ]:
# Visualize outliers BEFORE cleaning
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['age', 'gpa', 'expected_salary']):
    ax.boxplot(df[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='#3498db', color='black'))
    ax.set_title(f'{col} — Before Cleaning', fontweight='bold')
    ax.set_ylabel(col)
plt.suptitle('Outliers Before Treatment', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Fix invalid ranges
before_age = len(df)

df['age']              = df['age'].clip(lower=18, upper=35)
df['gpa']              = df['gpa'].clip(lower=0.0, upper=4.0)
df['experience_years'] = df['experience_years'].clip(lower=0, upper=10)
df['num_skills']       = df['num_skills'].clip(lower=0, upper=20)
df['expected_salary']  = df['expected_salary'].clip(lower=20000, upper=200000)

# Fix phone numbers — keep only valid ones (11 digits starting with 03)
df['phone_number'] = df['phone_number'].astype(str)
df['phone_valid']  = df['phone_number'].str.match(r'^03\d{9}$')
df.loc[~df['phone_valid'], 'phone_number'] = np.nan
df = df.drop(columns=['phone_valid'])

# Fix application_date — coerce invalid dates to NaT, then fill with most common date
df['application_date'] = pd.to_datetime(df['application_date'], errors='coerce')
df['application_date'] = df['application_date'].fillna(df['application_date'].mode()[0])

print(' Invalid values and outliers fixed!')
print(f'   Age range      : {df["age"].min()} – {df["age"].max()}')
print(f'   GPA range      : {df["gpa"].min()} – {df["gpa"].max()}')
print(f'   Salary range   : {df["expected_salary"].min()} – {df["expected_salary"].max()}')

In [ ]:
# Visualize AFTER cleaning
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['age', 'gpa', 'expected_salary']):
    ax.boxplot(df[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='#2ecc71', color='black'))
    ax.set_title(f'{col} — After Cleaning', fontweight='bold')
    ax.set_ylabel(col)
plt.suptitle('Outliers After Treatment', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 8: Data Type Correction & Feature Engineering

In [ ]:
# Correct data types
df['age']              = df['age'].astype(int)
df['experience_years'] = df['experience_years'].astype(int)
df['num_skills']       = df['num_skills'].astype(int)
df['expected_salary']  = df['expected_salary'].astype(int)
df['gpa']              = df['gpa'].round(2)

# Feature Engineering
df['application_year']  = df['application_date'].dt.year
df['application_month'] = df['application_date'].dt.month

# GPA Category
def gpa_category(g):
    if g >= 3.5: return 'Excellent'
    elif g >= 3.0: return 'Good'
    elif g >= 2.5: return 'Average'
    else: return 'Below Average'

df['gpa_category'] = df['gpa'].apply(gpa_category)

# Experience Category
df['experience_level'] = pd.cut(
    df['experience_years'],
    bins=[-1, 0, 2, 5, 10],
    labels=['Fresher', 'Junior', 'Mid-Level', 'Senior']
)

print(' Data types corrected!')
print(' New features added: application_year, application_month, gpa_category, experience_level')
print()
print(df.dtypes)

## Step 9: Before vs After Comparison

In [ ]:
print('CLEANING SUMMARY REPORT')
print('=' * 50)
print(f'  Original rows        : {df_raw.shape[0]}')
print(f'  Cleaned rows         : {df.shape[0]}')
print(f'  Duplicates removed   : {df_raw.shape[0] - df.shape[0]}')
print(f'  Missing values (raw) : {df_raw.isnull().sum().sum()}')
print(f'  Missing values (clean): {df.isnull().sum().sum()}')
print(f'  Columns (raw)        : {df_raw.shape[1]}')
print(f'  Columns (clean)      : {df.shape[1]}')
print(f'  New features added   : 4')
print('=' * 50)

In [ ]:
# Distribution charts after cleaning
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

df['department'].value_counts().plot(kind='bar', ax=axes[0,0], color='#3498db', edgecolor='white')
axes[0,0].set_title('Applications by Department', fontweight='bold')
axes[0,0].tick_params(axis='x', rotation=45)

df['gpa_category'].value_counts().plot(kind='pie', ax=axes[0,1],
    autopct='%1.1f%%', colors=['#2ecc71','#f39c12','#e74c3c','#9b59b6'])
axes[0,1].set_title('GPA Category Distribution', fontweight='bold')
axes[0,1].set_ylabel('')

df['experience_level'].value_counts().plot(kind='bar', ax=axes[1,0],
    color='#9b59b6', edgecolor='white')
axes[1,0].set_title('Experience Level Distribution', fontweight='bold')
axes[1,0].tick_params(axis='x', rotation=0)

axes[1,1].hist(df['expected_salary'], bins=20, color='#f39c12', edgecolor='white')
axes[1,1].set_title('Expected Salary Distribution', fontweight='bold')
axes[1,1].set_xlabel('Salary (PKR)')

plt.suptitle('Cleaned Data — Key Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 10: Export Cleaned Data

In [ ]:
df.to_csv('cleaned_applications.csv', index=False)

print(f' Cleaned data exported to cleaned_applications.csv')
print(f'   Final shape: {df.shape[0]} rows × {df.shape[1]} columns')
print()
df.head(10)